# Part 3: Constant Maturity Swaps

In [1]:
import pandas as pd
import numpy as np
from scipy.stats import norm
from scipy.optimize import brentq
import matplotlib.pylab as plt
from scipy.optimize import least_squares
import warnings
import pysabr
from pysabr import Hagan2002LognormalSABR

## CMS Products - Key Differences:
1. **Underlying**: CMS products are based on the swap rate, which is an average of forward rates over a specified period. In contrast, caplets and floorlets are based on a single forward rate at a specific time.
2. **Payoff Structure**: CMS products typically have a payoff that depends on the average swap rate over a period, while caplets and floorlets have payoffs that depend on the forward rate at a specific time.
3. **Volatility Modeling**: The volatility structure for CMS products can be more complex due to the averaging of rates, while caplets and floorlets can be modeled using simpler volatility structures.
4. **Calibration**: Calibrating models for CMS products may require more sophisticated techniques due to the complexity of the underlying swap rate, while caplets and floorlets can often be calibrated using standard methods.

## SABR Model Revisited

The implied Black volatility of the SABR model is given below, where $\beta = 0.75$ as a default setting

$$
    \begin{split}
      \sigma_{SABR}(F_0, K, \alpha, \beta, \rho, \nu)
      = \frac{\alpha}{(F_0K)^{(1-\beta)/2}\left\{ 1 + \frac{(1-\beta)^2}{24}\log^2\left(\frac{F_0}{K}\right) + \frac{(1-\beta)^4}{1920}\log^4\left(\frac{F_0}{K}\right) + \cdots\right\} }
      \times \frac{z}{x(z)} \times \left\{ 1 + \left[
           \frac{(1-\beta)^2}{24}
           \frac{\alpha^2}{(F_0K)^{1-\beta}}+\frac{1}{4}\frac{\rho\beta\nu\alpha}{(F_0K)^{(1-\beta)/2}}+\frac{2-3\rho^2}{24}\nu^2\right]
         T + \cdots \right.
    \end{split}
$$

where

$$
    \begin{split}
      z = \frac{\nu}{\alpha} (F_0K)^{(1-\beta)/2}
      \log\left(\frac{F_0}{K}\right),
    \end{split}
$$

and

$$
    % \begin{split}
      x(z) = \log \left[ \frac{\sqrt{1-2\rho z+z^2}+z -\rho}{1-\rho}
      \right].
    % \end{split}
$$

The definition above contains the function

$$
\begin{split}
{SABR}(F, K, T, \alpha, \beta, \rho, \nu)
\end{split}
$$


In [2]:
def SABR(F, K, T, alpha, beta, rho, nu):
    X = K
    # if K is at-the-money-forward
    if abs(F - K) < 1e-12:
        numer1 = (((1 - beta)**2)/24)*alpha*alpha/(F**(2 - 2*beta))
        numer2 = 0.25*rho*beta*nu*alpha/(F**(1 - beta))
        numer3 = ((2 - 3*rho*rho)/24)*nu*nu
        VolAtm = alpha*(1 + (numer1 + numer2 + numer3)*T)/(F**(1-beta))
        sabrsigma = VolAtm
    else:
        z = (nu/alpha)*((F*X)**(0.5*(1-beta)))*np.log(F/X)
        zhi = np.log((((1 - 2*rho*z + z*z)**0.5) + z - rho)/(1 - rho))
        numer1 = (((1 - beta)**2)/24)*((alpha*alpha)/((F*X)**(1 - beta)))
        numer2 = 0.25*rho*beta*nu*alpha/((F*X)**((1 - beta)/2))
        numer3 = ((2 - 3*rho*rho)/24)*nu*nu
        numer = alpha*(1 + (numer1 + numer2 + numer3)*T)*z
        denom1 = ((1 - beta)**2/24)*(np.log(F/X))**2
        denom2 = (((1 - beta)**4)/1920)*((np.log(F/X))**4)
        denom = ((F*X)**((1 - beta)/2))*(1 + denom1 + denom2)*zhi
        sabrsigma = numer/denom

    return sabrsigma

In [3]:
sabr_calibration_results = pd.read_csv('sabr_calibration_results.csv')

In [4]:
sabr_calibration_results

,Expiry,Tenor,Forward,alpha,rho,nu
0,1.0,1.0,0.033724,0.000910,-0.622993,0.023245
1,1.0,2.0,0.034525,0.001214,-0.499492,0.018643
2,1.0,3.0,0.035247,0.001278,-0.451604,0.015668
3,1.0,5.0,0.036608,0.001137,-0.377649,0.011376
4,1.0,10.0,0.039007,0.001087,-0.206444,0.008073
5,5.0,1.0,0.039468,0.001191,-0.553762,0.015110
6,5.0,2.0,0.039911,0.001227,-0.860808,0.015055
7,5.0,3.0,0.040339,0.001341,-0.504689,0.009262
8,5.0,5.0,0.041100,0.001198,-0.455872,0.006456
9,5.0,10.0,0.042281,0.001112,-0.355930,0.004643


## Calculating SABR model Present Values

1. PV of leg receiving CMS 10y semiannually over next 5 years
2. PV of leg receiving CMS 2y quarterly over next 10 years

## Forward Swap Rates vs CMS Rates

- 1 x 1, 1 x 10
- 5 x 1, 5 x 10
- 10 x 1, 10 x 10

Calculating this requires convexity correction, which can have a workaround via numerical integration